# Task 3 — Marketing Funnel & Conversion Performance Analysis

## 1. Data Loading & Sampling

The raw dataset (`2019-Oct.csv`) contains ~42.4M e-commerce events from October 2019.
Loading the full file is impractical, so we sample the **first 7 days** to preserve
complete sessions while keeping memory manageable.

In [1]:
import pandas as pd
from pandas.conftest import skipna

chunks = []
for chunk in pd.read_csv('../data/2019-Oct.csv', chunksize=1000000, parse_dates=["event_time"]):
    chunk = chunk[chunk["event_time"] < "2019-10-08"]
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print(f"Loaded {len(df):,} rows")

Loaded 8,829,315 rows


In [3]:
# Save sample df
df.to_parquet("../data/sample_first_week.parquet", index=False)

## 2. Initial Inspection

Basic shape, dtypes, and uniqueness checks on the sampled week.

In [6]:
import sys
sys.path.append("..")

import pandas as pd
from utilities.cleaning import report, check_categories

# Load sample df
df = pd.read_parquet("../data/sample_first_week.parquet")

%load_ext autoreload
%autoreload 2

In [11]:
report(df)

Total rows: 8829315, Total columns: 9

Duplicate rows: 5712

Missing values:
category_code: 2784814 missing values
brand: 1212750 missing values
user_session: 1 missing values

The first and last 5 rows of the dataset:
                 event_time event_type  product_id          category_id  \
0 2019-10-01 00:00:00+00:00       view    44600062  2103807459595387724   
1 2019-10-01 00:00:00+00:00       view     3900821  2053013552326770905   
2 2019-10-01 00:00:01+00:00       view    17200506  2053013559792632471   
3 2019-10-01 00:00:01+00:00       view     1307067  2053013558920217191   
4 2019-10-01 00:00:04+00:00       view     1004237  2053013555631882655   

                         category_code     brand    price    user_id  \
0                                 None  shiseido    35.79  541312140   
1  appliances.environment.water_heater      aqua    33.20  554748717   
2           furniture.living_room.sofa      None   543.10  519107250   
3                   computers.notebook    

In [12]:
check_categories(df)

Categories in event_type
event_type
view        8494427
cart         182773
purchase     152115
Name: count, dtype: int64

Categories in category_code
category_code
electronics.smartphone          2464145
electronics.clocks               302070
computers.notebook               261758
electronics.audio.headphone      236394
electronics.video.tv             206341
                                 ...   
apparel.skirt                       252
apparel.shoes.step_ins              246
apparel.shorts                      153
country_yard.furniture.bench         78
apparel.jacket                       35
Name: count, Length: 123, dtype: int64

Categories in brand
brand
samsung     1108595
apple        936647
xiaomi       632095
huawei       248560
lucente      120134
             ...   
magnat            1
unikum            1
crusader          1
staub             1
celebrat          1
Name: count, Length: 2639, dtype: int64

Categories in user_session
user_session
fb075266-182d-4c11-b5f7-4e4d

In [13]:
print(f"Unique users:    {df['user_id'].nunique():,}")
print(f"Unique sessions: {df['user_session'].nunique():,}")
print(f"Unique products: {df['product_id'].nunique():,}")
print(f"Date range:      {df['event_time'].min()} → {df['event_time'].max()}")

Unique users:    957,543
Unique sessions: 1,877,365
Unique products: 118,333
Date range:      2019-10-01 00:00:00+00:00 → 2019-10-07 23:59:59+00:00


## 3. Missing Value Analysis

Three columns have nulls: `category_code` (31.5%), `brand` (13.7%), and `user_session`
(1 row). Before deciding how to handle them, we diagnose whether the missingness is
random or systematic.

### 3.1 Is `category_id` also missing?

If only `category_code` (the string) is missing but `category_id` (the numeric ID)
is present, the codes may be recoverable via a lookup.

In [14]:
# Is category_id also missing?
print(df["category_id"].isnull().sum())

0


In [15]:
# Does missingness vary by event type?
print(df.groupby("event_type")["category_code"].apply(lambda x: x.isnull().mean()))

event_type
cart        0.093291
purchase    0.222831
view        0.321842
Name: category_code, dtype: float64


In [23]:
# Does missingness vary by brand?
brand_counts = df["brand"].value_counts()
big_brands = brand_counts[brand_counts > 1000].index

brand_code_miss = (
    df[df['brand'].isin(big_brands)].groupby('brand')['category_code'].apply(
        lambda x: x.isnull().mean()).sort_values(ascending=False)
)

print(brand_code_miss.head(10))   # worst 10
print(brand_code_miss.tail(10))   # best 10

brand
a-case       1.0
epr          1.0
erkaplan     1.0
erlit        1.0
roadx        1.0
farroad      1.0
roadstone    1.0
febest       1.0
rivertoys    1.0
rimax        1.0
Name: category_code, dtype: float64
brand
gree       0.0
rado       0.0
rals       0.0
rant       0.0
google     0.0
redbo      0.0
redford    0.0
gionee     0.0
reebok     0.0
zte        0.0
Name: category_code, dtype: float64
